In [ ]:
import torch
import numpy as np
import json
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

In [ ]:
which_code = "modular" # dino or supervised or modular
consider_same_patients = True

In [ ]:
if which_code == "dino":
  features = torch.load("./2-testfeat.pth")
  labels = torch.load("./2-testlabels.pth")
  with open('2-file_names.json') as f:
    file_names = json.load(f)
elif which_code == "supervised":
  features = np.load("./2-embeddings.npy")
  labels = np.load("./2-labels.npy")
  with open('2-test_file_names.json') as f:
    file_names = json.load(f)
elif which_code == "modular":
  features = np.load("./embeddings.npy")
  labels = np.load("./labels.npy")
  with open('file_paths.json') as f:
    file_names = json.load(f)
  file_names['test_file_names'] = file_names['test_file_paths']

In [ ]:
features = np.asarray(features)
labels = np.asarray(labels)
file_names = file_names['test_file_names']

if which_code == "supervised":
  for i, name in enumerate(file_names):
    file_names[i] = name.split("/")[-1]

In [ ]:
print(features.shape)
print(labels.shape)
print(len(file_names))

(3000, 256)
(3000,)
3000


In [ ]:
total_number = 0
currect_number = 0

for i in tqdm(range(len(file_names))):
  current_vector = features[i, :]
  current_label = int(labels[i])
  current_patient_id = int(file_names[i].split(".")[0].split("_")[-1])

  knn = NearestNeighbors(n_neighbors=len(features), metric="euclidean")
  knn.fit(features)
  distances, indices = knn.kneighbors([current_vector])
  sorted_indices = indices.flatten()
  sorted_indices = sorted_indices[1:]

  if consider_same_patients:
    for neighbour in sorted_indices:
      neighbour_patient_id = int(file_names[neighbour].split(".")[0].split("_")[-1])
      if current_patient_id != neighbour_patient_id:
        neighbour_label = int(labels[neighbour])
        if current_label == neighbour_label:
          currect_number += 1
        break
  else:
    nearest_neighbour = sorted_indices[0]
    nearest_label = int(labels[nearest_neighbour])
    if current_label == nearest_label:
      currect_number += 1

  total_number += 1

100%|██████████| 3000/3000 [00:11<00:00, 267.98it/s]


In [ ]:
currect_number / total_number

0.3253333333333333